# Explore PARLO

This notebook distinguishes **participants** from **recordings** throughout. Metadata and text queries do not decode audio. Edit the corpus root below, then run all cells. Optional adapters run only when their extras are installed.

The paper's curated subsets differ from file availability in this download. Placeholder CHAT files are retained for audio exploration and excluded from ASR references. See the package README for the task definitions and transcription limitations.

In [ ]:
from pathlib import Path
CORPUS_ROOT = Path('../parlo_dataset')  # the only setting to edit; relative to this notebook

import json
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display, Audio as AudioPlayer, clear_output
from parlo import Corpus, SplitManifest, iter_rows

corpus = Corpus(CORPUS_ROOT)
print('Cache:', corpus.cache_status())
recordings = corpus.recordings
participants = corpus.participants
print('Participants in metadata:', participants.select(pl.len()).collect().item())
print('Participants with recordings:', participants.filter(pl.col('has_recordings')).select(pl.len()).collect().item())
print('Paired recordings:', recordings.select(pl.len()).collect().item())

## A lazy query

`explain()` shows the query plan. `.collect()` reads the requested table data, never waveforms. Metadata strings preserve the original CSV representation; clinical and demographic summaries cast explicitly.

In [ ]:
query = recordings.group_by('task', 'diagnosis').agg(
    pl.len().alias('recording_count'),
    pl.col('participant_id').n_unique().alias('participant_count'),
    (pl.col('duration_ms').sum() / 60000).alias('audio_minutes'),
).sort('task', 'diagnosis')
print(query.explain())
display(query.collect())

## Task, group, and participant selectors

Selectors update a small preview. The audio button decodes only the first eligible participant utterance in the current selection. No audio loads when selectors change.

In [ ]:
import ipywidgets as widgets

tasks = recordings.select('task').unique().sort('task').collect()['task'].to_list()
task_selector = widgets.Dropdown(options=['All'] + tasks, description='Task:')
group_selector = widgets.Dropdown(options=['All', 'HC', 'MCI', 'AD'], description='Group:')
participant_selector = widgets.Dropdown(options=['All'], description='Participant:')
preview = widgets.Output()
playback = widgets.Output()
play_button = widgets.Button(description='Load selected excerpt', icon='play')

def selected_view(include_participant=True):
    view = corpus.filter(
        task=None if task_selector.value == 'All' else task_selector.value,
        diagnosis=None if group_selector.value == 'All' else group_selector.value,
    )
    if include_participant and participant_selector.value != 'All':
        view = view.participant(participant_selector.value)
    return view

def show_selection(change=None):
    view = selected_view()
    with preview:
        clear_output(wait=True)
        print('Selected participants:', view.participants.select(pl.len()).collect().item())
        print('Selected recordings:', view.recordings.select(pl.len()).collect().item())
        display(view.recordings.select('participant_id', 'diagnosis', 'task', 'duration_ms', 'transcript_status').limit(8).collect())
    play_button.disabled = view.asr().limit(1).collect().is_empty()

def update_participants(change=None):
    ids = selected_view(False).participants.select('participant_id').sort('participant_id').collect()['participant_id'].to_list()
    previous = participant_selector.value
    participant_selector.options = ['All'] + ids
    participant_selector.value = previous if previous in participant_selector.options else 'All'
    show_selection()

def play_selected(button):
    with playback:
        clear_output(wait=True)
        examples = selected_view().asr().sort('utterance_id').limit(1).collect()
        if examples.is_empty():
            print('No timed participant reference in this selection.')
            return
        example = examples.row(0, named=True)
        audio = corpus.load_audio(example, mono=True)
        print(example['utterance_id'], example['reference_verbatim'])
        display(AudioPlayer(audio.samples, rate=audio.sample_rate))

task_selector.observe(update_participants, names='value')
group_selector.observe(update_participants, names='value')
participant_selector.observe(show_selection, names='value')
play_button.on_click(play_selected)
update_participants()
display(widgets.VBox([widgets.HBox([task_selector, group_selector, participant_selector]), preview, play_button, playback]))

## Participant demographics and clinical scores

Each participant contributes once to these summaries, regardless of how many recordings they have. Missing values remain missing; CDR is never filled with zero.

In [ ]:
numeric = participants.with_columns([
    pl.col(name).cast(pl.Float64, strict=False).alias(name)
    for name in ['age', 'totalYearsofEducation', 'mmse_value', 'cdr_value']
])
demographics = numeric.group_by('diagnosis').agg(
    pl.len().alias('participant_count'),
    pl.col('age').mean().alias('mean_age'),
    pl.col('totalYearsofEducation').mean().alias('mean_education_years'),
    pl.col('mmse_value').mean().alias('mean_mmse'),
    pl.col('mmse_value').null_count().alias('participants_missing_mmse'),
    pl.col('cdr_value').mean().alias('mean_cdr'),
    pl.col('cdr_value').null_count().alias('participants_missing_cdr'),
).sort('diagnosis').collect()
display(demographics)
display(participants.group_by('diagnosis', 'gender').len(name='participant_count').sort('diagnosis', 'gender').collect())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for group in ['HC', 'MCI', 'AD']:
    subset = numeric.filter(pl.col('diagnosis') == group).collect()
    axes[0].hist(subset['age'].drop_nulls().to_list(), bins=10, alpha=.45, label=group)
    axes[1].hist(subset['mmse_value'].drop_nulls().to_list(), bins=10, alpha=.45, label=group)
for ax, label in zip(axes, ['Age (years)', 'MMSE score']):
    ax.set(xlabel=label, ylabel='Participants')
    ax.legend()
fig.tight_layout()
plt.show()

## Task availability, duration, and transcript coverage

These plots count recordings, not independent participants. A file pair may carry a placeholder transcript. The supplied download has 201 participants with all eight pairs; the paper reports 171 in its curated all-task subset.

In [ ]:
coverage = recordings.group_by('task').agg(
    pl.len().alias('recordings'),
    pl.col('participant_id').n_unique().alias('participants'),
    (pl.col('transcript_status') == 'placeholder').sum().alias('placeholders'),
    pl.col('valid_reference').sum().alias('recording_asr_eligible'),
    (pl.col('duration_ms').mean() / 1000).alias('mean_seconds'),
).sort('task').collect()
display(coverage)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(coverage['task'], coverage['recordings'], label='Available pairs', color='#628aa7')
axes[0].bar(coverage['task'], coverage['recording_asr_eligible'], label='ASR eligible', color='#2b596e')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set(ylabel='Recordings', title='File availability and reference coverage')
axes[0].legend()
for group in ['HC', 'MCI', 'AD']:
    durations = recordings.filter(pl.col('diagnosis') == group).select(pl.col('duration_ms') / 1000).collect().to_series()
    axes[1].hist(durations, bins=25, alpha=.4, label=group)
axes[1].set(xlabel='Duration (seconds)', ylabel='Recordings')
axes[1].legend()
fig.tight_layout()
plt.show()

## Original text, speakers, annotations, and timing

Inspect participant turns and interviewer turns separately. Raw CHAT, headers, and dependent tiers remain available. Integer milliseconds are measured from recording start. Missing or invalid annotations are not repaired.

In [ ]:
examples = corpus.asr().sort('utterance_id').limit(1).collect()
example = examples.row(0, named=True) if not examples.is_empty() else None
if example:
    recording = corpus.recordings.filter(pl.col('recording_id') == example['recording_id']).collect().row(0, named=True)
    display(corpus.utterances.filter(pl.col('recording_id') == example['recording_id']).select(
        'speaker', 'is_participant', 'text', 'start_ms', 'end_ms', 'timing_status').collect())
    print('Raw participant tier:', example['raw_text'])
    print('Dependent tiers:', json.loads(example['dependent_tiers']))
    print('Headers:', json.loads(recording['headers']))
    display(corpus.words.filter(pl.col('utterance_id') == example['utterance_id']).select(
        'text', 'start_ms', 'end_ms', 'timing_status').limit(15).collect())
display(corpus.utterances.filter(~pl.col('is_participant')).select('recording_id', 'speaker', 'text').limit(5).collect())
display(corpus.validation_report.collect())

## Explicit playback

Running this cell requests one excerpt. Other cells do not decode audio; the selectors above also provide a playback button. Native rate is preserved, with mono requested for the player.

In [ ]:
if example:
    audio = corpus.load_audio(example, mono=True)
    print(f'{audio.sample_rate} Hz, {len(audio.samples)} samples; interval {example["start_ms"]}–{example["end_ms"]} ms')
    display(AudioPlayer(audio.samples, rate=audio.sample_rate))

## Multiple tasks and incomplete cases

Default filtering keeps every available recording. Complete-case filtering is explicit and describes file availability; it does not require valid transcripts. Clinical targets apply their own missing-value exclusions.

In [ ]:
multi = corpus.filter(task=['reading', 'recall'])
complete = multi.complete_cases(['reading', 'recall'])
print('Available-task participants:', multi.participants.select(pl.len()).collect().item())
print('Participants with both pairs:', complete.participants.select(pl.len()).collect().item())
display(multi.recordings.group_by('participant_id').agg(pl.col('task').unique().sort().alias('available_tasks')).filter(
    pl.col('available_tasks').list.len() < 2).collect())
print('All-eight-pair participants:', corpus.complete_cases(tasks).participants.select(pl.len()).collect().item())
if example:
    display(corpus.participant(example['participant_id']).recordings.select('task', 'transcript_status').collect())

## Reusable participant splits and leakage checks

Generate assignments on the full cohort first. Save only participant IDs, then reuse the manifest for any task or segment view. Default seed 42, train/validation/test proportions 70/15/15. Infeasible small stratified groups raise actionable errors.

In [ ]:
split = corpus.split()
split_path = corpus.cache_dir / 'example-participant-split.json'
split.save(split_path)
split = SplitManifest.load(split_path)
partitions = {name: set(ids) for name, ids in split.assignments.items()}
for i, name in enumerate(partitions):
    for other in list(partitions)[i + 1:]:
        assert partitions[name].isdisjoint(partitions[other])
for table in [multi.recordings, corpus.asr()]:
    expanded = [set(split.apply(table, name).select('participant_id').collect().to_series()) for name in partitions]
    assert all(a.isdisjoint(b) for i, a in enumerate(expanded) for b in expanded[i + 1:])
print('Participant split sizes:', {name: len(ids) for name, ids in partitions.items()})
print('No participant leakage across recordings or ASR utterances.')
folds = corpus.kfold()
held_out = [pid for fold in folds for pid in fold.assignments['test']]
assert len(held_out) == len(set(held_out))
print('Five-fold participant test sizes:', [len(f.assignments['test']) for f in folds])
print('Leave-one-participant-out folds:', sum(1 for _ in corpus.leave_one_out()))

## Classification and regression views

Original labels remain accessible. Pairwise target numbers follow the requested label order. Missing regression scores exclude participants only from that target view and are reported explicitly.

In [ ]:
print('Three-way mapping:', corpus.targets().mapping)
print('HC/AD mapping:', corpus.targets(labels=['HC', 'AD']).mapping)
print('HC/impaired mapping:', corpus.targets('hc_vs_impaired').mapping)
for target in ['mmse', 'cdr']:
    target_view = corpus.targets(target)
    print(target.upper(), 'eligible participants:', target_view.participants.select(pl.len()).collect().item())
    display(target_view.exclusions.collect())
    display(target_view.expand('recordings').select('participant_id', 'task', 'diagnosis', 'target').limit(3).collect())

## ASR preparation and metadata-only shuffling

Timed utterance references contain participant speech. Full-recording references include every speaker, matching the full audio. Both expose cleaned verbatim references and configurable normalization; fillers and repetitions are retained. This notebook does not run ASR or compute WER.

Index shuffling materializes the metadata rows and a random index permutation. Buffer shuffling stores only a limited row window plus batches; it is not a uniform global shuffle.

In [ ]:
asr = corpus.asr()
display(asr.select('utterance_id', 'audio_path', 'start_ms', 'end_ms', 'reference_verbatim', 'reference').limit(3).collect())
print('Timed participant ASR examples:', asr.select(pl.len()).collect().item())
print('Full-recording ASR examples:', corpus.asr('recording').select(pl.len()).collect().item())
assert asr.filter(pl.col('is_placeholder') | (pl.col('timing_status') != 'valid')).collect().is_empty()
for batch in corpus.iter_batches('recordings', batch_size=128):
    assert batch.height <= 128
# Stable sort is useful for reproducible shuffling of arbitrary query plans.
small = asr.sort('utterance_id').limit(10)
for mode in ['index', 'buffer']:
    shuffled = list(iter_rows(small, shuffle=mode, seed=42, buffer_size=3))
    print(mode, [r['utterance_id'] for r in shuffled[:3]])

## Optional PyTorch and Hugging Face adapters

These return dictionaries containing relative paths and timing fields. Creating or iterating them does not decode audio. Explicitly call `corpus.load_audio(row)` at the point your application needs samples. The map adapters materialize metadata; iterable adapters stream it.

In [ ]:
import importlib.util
train_query = split.apply(asr, 'train').sort('utterance_id')
if importlib.util.find_spec('torch'):
    from parlo.torch import ParloDataset, ParloIterableDataset
    torch_rows = ParloDataset(train_query)
    torch_stream = ParloIterableDataset(train_query, shuffle='buffer', seed=42)
    print('Torch metadata rows:', len(torch_rows))
    if len(torch_rows):
        print('Deferred audio path:', torch_rows[0]['audio_path'])
        print('First streamed row:', next(iter(torch_stream))['utterance_id'])
else:
    print('Optional: uv pip install "parlo-loader[torch]"')
if importlib.util.find_spec('datasets'):
    from parlo.huggingface import to_huggingface
    hf_rows = to_huggingface(train_query)
    hf_stream = to_huggingface(train_query, iterable=True)
    print('Hugging Face metadata rows:', len(hf_rows))
    if len(hf_rows):
        print('Deferred audio path:', next(iter(hf_stream))['audio_path'])
else:
    print('Optional: uv pip install "parlo-loader[huggingface]"')

## Cache status and interpretation

The initial cache stores text/metadata and WAV headers, not decoded audio. A new `Corpus` instance checks file paths, sizes, and modification times. Call `corpus.refresh(content_hash=True)` to establish SHA-256 hashes when byte-level verification is needed; it reads every source file. `corpus.cache_status(verify_hashes=True)` checks that baseline without rebuilding.

The live Corpus is a snapshot. After a refresh, create new views. Keep caches local: they contain corpus material. File availability, ASR eligibility, and the paper's curated samples describe different selections.

In [ ]:
print(corpus.cache_status())
display(corpus.validation_report.group_by('code').len(name='issue_count').collect())